# Notebook 1: Carga y exploración de los datos

**Objetivo:** cargar las dos fuentes del proyecto (`bank-additional.csv` y `customer-details.xlsx`) y hacer una primera exploración para detectar qué hay que limpiar. Aquí no se modifica ningún dato.

## Preparación
Importo las librerías y configuro pandas para que muestre todas las columnas.

In [1]:
import pandas as pd
import numpy as np

pd.set_option ("display.max_columns", None)

## 1. Carga de `bank-additional.csv`
Datos de las llamadas de una campaña de marketing de un banco portugués. Cada fila es un cliente contactado. Lo cargo en el DataFrame `bank`.

In [2]:
bank = pd.read_csv("../DatosProyecto/bank-additional.csv")

bank.head()



,Unnamed: 0,age,job,marital,education,default,housing,loan,contact,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,date,latitude,longitude,id_
0,0,NaN,housemaid,MARRIED,basic.4y,0.0,0.0,0.0,telephone,261,1,999,0,NONEXISTENT,1.1,"93,994","-36,4","4,857",5191,no,2-agosto-2019,41.495,-71.233,089b39d8-e4d0-461b-87d4-814d71e0e079
1,1,57.0,services,MARRIED,high.school,NaN,0.0,0.0,telephone,149,1,999,0,NONEXISTENT,1.1,"93,994","-36,4",NaN,5191,no,14-septiembre-2016,34.601,-83.923,e9d37224-cb6f-4942-98d7-46672963d097
2,2,37.0,services,MARRIED,high.school,0.0,1.0,0.0,telephone,226,1,999,0,NONEXISTENT,1.1,"93,994","-36,4","4,857",5191,no,15-febrero-2019,34.939,-94.847,3f9f49b5-e410-4948-bf6e-f9244f04918b
3,3,40.0,admin.,MARRIED,basic.6y,0.0,0.0,0.0,telephone,151,1,999,0,NONEXISTENT,1.1,"93,994","-36,4",NaN,5191,no,29-noviembre-2015,49.041,-70.308,9991fafb-4447-451a-8be2-b0df6098d13e
4,4,56.0,services,MARRIED,high.school,0.0,0.0,1.0,telephone,307,1,999,0,NONEXISTENT,1.1,"93,994","-36,4",NaN,5191,no,29-enero-2017,38.033,-104.463,eca60b76-70b6-4077-80ba-bc52e8ebb0eb


## 2. Carga de `customer-details.xlsx`
Datos demográficos de los clientes del banco. Es un Excel con tres hojas (una por cada año de alta: 2012, 2013 y 2014), por lo que se cargan con `read_excel`, no con `read_csv`. Ya los uniremos más adelante.

In [3]:
customer_2012 = pd.read_excel("../DatosProyecto/customer-details.xlsx", sheet_name="2012")
customer_2013 = pd.read_excel("../DatosProyecto/customer-details.xlsx", sheet_name="2013")
customer_2014 = pd.read_excel("../DatosProyecto/customer-details.xlsx", sheet_name="2014")

customer_2012.head()

,Unnamed: 0,Income,Kidhome,Teenhome,Dt_Customer,NumWebVisitsMonth,ID
0,0,161770,1,0,2012-04-04,29,089b39d8-e4d0-461b-87d4-814d71e0e079
1,1,85477,1,1,2012-12-30,7,e9d37224-cb6f-4942-98d7-46672963d097
2,2,147233,1,1,2012-02-02,5,3f9f49b5-e410-4948-bf6e-f9244f04918b
3,3,121393,1,2,2012-12-21,29,9991fafb-4447-451a-8be2-b0df6098d13e
4,4,63164,1,2,2012-06-20,20,eca60b76-70b6-4077-80ba-bc52e8ebb0eb


## 3. Exploración de `bank`

### Tamaño
`bank` tiene **43.000 filas (clientes) y 24 columnas**.

In [4]:
bank.shape

(43000, 24)

### Tipos de datos y nulos
Con `info()` obtengo los tipos y los valores no nulos de cada columna. Hay nulos en 10 columnas, y varias columnas tienen un tipo que no es el esperado (se verá en el apartado siguiente).

In [5]:
bank.info()

<class 'pandas.DataFrame'>
RangeIndex: 43000 entries, 0 to 42999
Data columns (total 24 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Unnamed: 0      43000 non-null  int64  
 1   age             37880 non-null  float64
 2   job             42655 non-null  str    
 3   marital         42915 non-null  str    
 4   education       41193 non-null  str    
 5   default         34019 non-null  float64
 6   housing         41974 non-null  float64
 7   loan            41974 non-null  float64
 8   contact         43000 non-null  str    
 9   duration        43000 non-null  int64  
 10  campaign        43000 non-null  int64  
 11  pdays           43000 non-null  int64  
 12  previous        43000 non-null  int64  
 13  poutcome        43000 non-null  str    
 14  emp.var.rate    43000 non-null  float64
 15  cons.price.idx  42529 non-null  str    
 16  cons.conf.idx   43000 non-null  str    
 17  euribor3m       33744 non-null  str    
 1

### Nombres de las columnas
Comparo las columnas con las del enunciado:
- `Unnamed: 0` es un índice sobrante del archivo.
- `contact_month` y `contact_year` aparecen en el enunciado pero no existen, así que tendré que crearlas a partir de `date`.

In [6]:
bank.dtypes

Unnamed: 0          int64
age               float64
job                   str
marital               str
education             str
default           float64
housing           float64
loan              float64
contact               str
duration            int64
campaign            int64
pdays               int64
previous            int64
poutcome              str
emp.var.rate      float64
cons.price.idx        str
cons.conf.idx         str
euribor3m             str
nr.employed           str
y                     str
date                  str
latitude          float64
longitude         float64
id_                   str
dtype: object

### Valores nulos
Hay nulos en 10 columnas. Las que más tienen son `euribor3m` (9.256), `default` (8.981), `age` (5.120) y `education` (1.807). Tampoco se quedan atrás `housing` y `loan` (1026)

In [7]:
bank.isna().sum().reset_index()

,index,0
0,Unnamed: 0,0
1,age,5120
2,job,345
3,marital,85
4,education,1807
5,default,8981
6,housing,1026
7,loan,1026
8,contact,0
9,duration,0


### Porcentaje de nulos
El porcentaje sobre las 43.000 filas decide el tratamiento:
- **Más del 20 %:** `euribor3m` (21,5 %) y `default` (20,9 %).
- **Alrededor del 12 %:** `age`.
- **Entre el 1 % y el 5 %:** `education`, `housing`, `loan` y `cons.price.idx`.
- **Menos del 1 %:** `job`, `date` y `marital`.

En la limpieza decidiré, columna a columna, si relleno, elimino o dejo los nulos.


In [8]:
media_valores_nulos = bank.isna().mean()*100
media_valores_nulos.round(2).reset_index()

,index,0
0,Unnamed: 0,0.00
1,age,11.91
2,job,0.80
3,marital,0.20
4,education,4.20
5,default,20.89
6,housing,2.39
7,loan,2.39
8,contact,0.00
9,duration,0.00


### Duplicados
Vemos que no hay duplicados en la hoja de `bank`

In [9]:
bank.duplicated().sum()

np.int64(0)

### Resumen estadístico
Con `describe()` reviso mínimos, máximos y medias de las columnas numéricas. Conclusiones:

- **age**: La edad media de los clientes del banco es casi de 40 años. Teniendo un máximo de 98 años y un mínimo de 17 años.
- **housing**: Algo más de la mitad (casi un 54%) de los clientes disponen de un préstamo hipotecario.
- **duration**: La media es de 257 segundos (unos 4 minutos). El mínimo de 0 segundos sugiere llamadas que no llegaron a establecerse, y el máximo de 4.918 segundos (unos 82 minutos) está muy lejos del percentil 75 (319 segundos), habría que revisar esta parte.

In [10]:
bank.describe().reset_index()

,index,Unnamed: 0,age,default,housing,loan,duration,campaign,pdays,previous,emp.var.rate,latitude,longitude
0,count,43000.000000,37880.000000,34019.000000,41974.000000,41974.000000,43000.000000,43000.000000,43000.000000,43000.000000,43000.000000,43000.000000,43000.000000
1,mean,20651.099047,39.977112,0.000088,0.535998,0.155620,257.739279,2.567233,962.330953,0.174023,0.077128,36.856697,-95.939067
2,std,11868.770913,10.437957,0.009390,0.498708,0.362499,258.666033,2.772294,187.260394,0.497366,1.573898,7.225948,16.752282
3,min,0.000000,17.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,-3.400000,24.396000,-124.997000
4,25%,10381.750000,32.000000,0.000000,0.000000,0.000000,102.000000,1.000000,999.000000,0.000000,-1.800000,30.614750,-110.494250
5,50%,20642.500000,38.000000,0.000000,1.000000,0.000000,179.000000,2.000000,999.000000,0.000000,1.100000,36.761000,-95.899500
6,75%,30930.250000,47.000000,0.000000,1.000000,0.000000,319.000000,3.000000,999.000000,0.000000,1.400000,43.113250,-81.427750
7,max,41187.000000,98.000000,1.000000,1.000000,1.000000,4918.000000,56.000000,999.000000,7.000000,1.400000,49.384000,-66.937000


### Columnas de categorías
Reviso con `value_counts()` cómo están escritos los valores y cuántos nulos hay:
- `marital` y `poutcome` están escritos en mayúsculas, y el resto de columnas de texto en minúsculas. Habrá que unificarlo.
- `job` (345), `marital` (85) y `education` (1.807) tienen nulos.
- `housing` y `loan` deberían valer solo 0 o 1, y tienen 1.026 nulos cada una.


In [11]:
columnas = ["job" , "marital", "education", "contact", "poutcome", "housing", "loan", "y"]

for col in columnas:
    print(col)
    print (bank[col].value_counts(dropna=False).reset_index())

job
              job  count
0          admin.  10873
1     blue-collar   9654
2      technician   7026
3        services   4162
4      management   3050
5         retired   1790
6    entrepreneur   1522
7   self-employed   1489
8       housemaid   1123
9      unemployed   1063
10        student    903
11            NaN    345
marital
    marital  count
0   MARRIED  25999
1    SINGLE  12105
2  DIVORCED   4811
3       NaN     85
education
             education  count
0    university.degree  12722
1          high.school   9925
2             basic.9y   6309
3  professional.course   5477
4             basic.4y   4356
5             basic.6y   2386
6                  NaN   1807
7           illiterate     18
contact
     contact  count
0   cellular  27396
1  telephone  15604
poutcome
      poutcome  count
0  NONEXISTENT  37103
1      FAILURE   4461
2      SUCCESS   1436
housing
   housing  count
0      1.0  22498
1      0.0  19476
2      NaN   1026
loan
   loan  count
0   0.0  35442
1   1.0 

### Columna `default`
`default` indica si el cliente tiene historial de **impago**. De 43.000 filas, 34.016 valen 0, solo 3 valen 1 y 8.981 son nulos (21 %).


In [12]:
bank["default"].value_counts(dropna=False)

default
0.0    34016
NaN     8981
1.0        3
Name: count, dtype: int64

## 4. Exploración de `customer`

### Tamaño
`customer_2012` tiene **20115 filas (clientes) y 7 columnas**.
`customer_2013` tiene **8965 filas (clientes) y 7 columnas**.
`customer_2014` tiene **14090 filas (clientes) y 7 columnas**.

In [13]:
customer_2012.shape

(20115, 7)

In [14]:
customer_2013.shape

(8965, 7)

In [15]:
customer_2014.shape

(14090, 7)

### Tipos de datos y nulos
Con `info()` obtengo los tipos y los valores no nulos de cada columna. En este caso parece que no hay ningún valor nulo. Además confirmamos que las 7 columnas que hay en cada hoja coinciden.

In [16]:
customer_2012.info()

<class 'pandas.DataFrame'>
RangeIndex: 20115 entries, 0 to 20114
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Unnamed: 0         20115 non-null  int64         
 1   Income             20115 non-null  int64         
 2   Kidhome            20115 non-null  int64         
 3   Teenhome           20115 non-null  int64         
 4   Dt_Customer        20115 non-null  datetime64[us]
 5   NumWebVisitsMonth  20115 non-null  int64         
 6   ID                 20115 non-null  str           
dtypes: datetime64[us](1), int64(5), str(1)
memory usage: 1.1 MB


In [17]:
customer_2013.info()

<class 'pandas.DataFrame'>
RangeIndex: 8965 entries, 0 to 8964
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Unnamed: 0         8965 non-null   int64         
 1   Income             8965 non-null   int64         
 2   Kidhome            8965 non-null   int64         
 3   Teenhome           8965 non-null   int64         
 4   Dt_Customer        8965 non-null   datetime64[us]
 5   NumWebVisitsMonth  8965 non-null   int64         
 6   ID                 8965 non-null   str           
dtypes: datetime64[us](1), int64(5), str(1)
memory usage: 490.4 KB


In [18]:
customer_2014.info()

<class 'pandas.DataFrame'>
RangeIndex: 14090 entries, 0 to 14089
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Unnamed: 0         14090 non-null  int64         
 1   Income             14090 non-null  int64         
 2   Kidhome            14090 non-null  int64         
 3   Teenhome           14090 non-null  int64         
 4   Dt_Customer        14090 non-null  datetime64[us]
 5   NumWebVisitsMonth  14090 non-null  int64         
 6   ID                 14090 non-null  str           
dtypes: datetime64[us](1), int64(5), str(1)
memory usage: 770.7 KB


### Valores nulos
En este caso no hay filas con valores nulos, lo revisamos de todas formas

In [19]:
customer_2012.isna().sum()

Unnamed: 0           0
Income               0
Kidhome              0
Teenhome             0
Dt_Customer          0
NumWebVisitsMonth    0
ID                   0
dtype: int64

In [20]:
customer_2013.isna().sum()

Unnamed: 0           0
Income               0
Kidhome              0
Teenhome             0
Dt_Customer          0
NumWebVisitsMonth    0
ID                   0
dtype: int64

In [21]:
customer_2014.isna().sum()

Unnamed: 0           0
Income               0
Kidhome              0
Teenhome             0
Dt_Customer          0
NumWebVisitsMonth    0
ID                   0
dtype: int64

### Duplicados

Revisamos que tampoco hay ninguna fila duplicada. Podemos tanto ver hoja por hoja o más rapido agrupar las 3 hojas y hacer un bucle.

In [22]:
customer_2012.duplicated().sum()

np.int64(0)

In [23]:
customer_2013.duplicated().sum()

np.int64(0)

In [24]:
customer_2014.duplicated().sum()

np.int64(0)

In [25]:
customer_total = [customer_2012, customer_2013, customer_2014]

for c in customer_total:
    print(c.duplicated().sum())


0
0
0


### Resumen estadístico
Con `describe()` reviso mínimos, máximos, medias y medianas de las columnas numéricas de cada hoja. Las tres hojas (2012, 2013 y 2014) dan resultados muy parecidos. Conclusiones:

- **Kidhome**: la media es de 1 niño pequeño por hogar y el máximo es 2.
- **Teenhome**: ocurre lo mismo que en la anterior. La media es de 1 adolescente por hogar, el máximo es 2.
- **NumWebVisitsMonth**: la media se sitúa en unas 16-17 visitas mensuales a la web, con un mínimo de 1 y un máximo de 32.
- **Income**: el ingreso anual mínimo es inferior a 6.000 y el máximo está en torno a 180.000. La media (unos 93.000) y la mediana son prácticamente iguales, por lo que los ingresos están repartidos de forma equilibrada, sin unos pocos valores muy altos que tiren de la media. Las tres hojas tienen valores muy similares.


In [26]:
customer_2012.describe().reset_index()

,index,Unnamed: 0,Income,Kidhome,Teenhome,Dt_Customer,NumWebVisitsMonth
0,count,20115.000000,20115.000000,20115.000000,20115.000000,20115,20115.000000
1,mean,10057.000000,93087.213224,1.004176,0.999503,2012-07-01 17:15:14.362416,16.540790
2,min,0.000000,5852.000000,0.000000,0.000000,2012-01-01 00:00:00,1.000000
3,25%,5028.500000,49110.000000,0.000000,0.000000,2012-04-01 00:00:00,9.000000
4,50%,10057.000000,92962.000000,1.000000,1.000000,2012-07-02 00:00:00,16.000000
5,75%,15085.500000,136705.000000,2.000000,2.000000,2012-09-30 00:00:00,25.000000
6,max,20114.000000,180791.000000,2.000000,2.000000,2012-12-31 00:00:00,32.000000
7,std,5806.844668,50614.798255,0.816811,0.815298,NaN,9.235324


In [27]:
customer_2013.describe().reset_index()

,index,Unnamed: 0,Income,Kidhome,Teenhome,Dt_Customer,NumWebVisitsMonth
0,count,8965.000000,8965.000000,8965.000000,8965.000000,8965,8965.000000
1,mean,4482.000000,92788.951255,0.991523,0.995761,2013-07-02 11:23:37.110987,16.575683
2,min,0.000000,5848.000000,0.000000,0.000000,2013-01-01 00:00:00,1.000000
3,25%,2241.000000,49866.000000,0.000000,0.000000,2013-04-01 00:00:00,9.000000
4,50%,4482.000000,92315.000000,1.000000,1.000000,2013-07-03 00:00:00,17.000000
5,75%,6723.000000,136063.000000,2.000000,2.000000,2013-10-03 00:00:00,25.000000
6,max,8964.000000,180787.000000,2.000000,2.000000,2013-12-31 00:00:00,32.000000
7,std,2588.116915,50188.145448,0.810005,0.814981,NaN,9.259737


In [28]:
customer_2014.describe().reset_index()

,index,Unnamed: 0,Income,Kidhome,Teenhome,Dt_Customer,NumWebVisitsMonth
0,count,14090.000000,14090.000000,14090.000000,14090.000000,14090,14090.000000
1,mean,7044.500000,93706.470405,1.013982,0.999219,2014-07-01 16:25:06.430092,16.673314
2,min,0.000000,5841.000000,0.000000,0.000000,2014-01-01 00:00:00,1.000000
3,25%,3522.250000,50180.250000,0.000000,0.000000,2014-04-04 00:00:00,9.000000
4,50%,7044.500000,93589.000000,1.000000,1.000000,2014-06-30 00:00:00,17.000000
5,75%,10566.750000,137195.500000,2.000000,2.000000,2014-09-30 00:00:00,25.000000
6,max,14089.000000,180802.000000,2.000000,2.000000,2014-12-31 00:00:00,32.000000
7,std,4067.576981,50527.970173,0.818562,0.817727,NaN,9.239609
